In [2]:

import numpy as np

class ContinuousGameSolver:
    def __init__(self, a, b, c, d, e):
        self.a = a
        self.b = b
        self.c = c
        self.d = d
        self.e = e

    def H(self, x, y):
        return self.a*x**2 + self.b*y**2 + self.c*x*y + self.d*x + self.e*y

    def analytical_solution(self):
        # Коэффициенты системы линейных уравнений:
        # 2a*x + c*y = -d
        # c*x + 2b*y = -e
        A1, B1, C1 = 2*self.a, self.c, -self.d   # первое уравнение: A1*x + B1*y = C1
        A2, B2, C2 = self.c, 2*self.b, -self.e   # второе уравнение: A2*x + B2*y = C2

        # Определитель системы
        det = A1*B2 - A2*B1

        # Решение по правилу Крамера
        x_star = (C1*B2 - C2*B1) / det
        y_star = (A1*C2 - A2*C1) / det

        x_star = max(0, min(1, x_star))
        y_star = max(0, min(1, y_star))

        # Цена игры
        v_star = self.H(x_star, y_star)

        print(f"\nЯдро игры: H(x,y) = {self.a}·x² + {self.b}·y² + {self.c}·x·y + {self.d}·x + {self.e}·y")
        print(f"\nОптимальная стратегия игрока A: x* = {x_star:.6f}")
        print(f"Оптимальная стратегия игрока B: y* = {y_star:.6f}")
        print(f"Цена игры: v = {v_star:.6f}")

        return x_star, y_star, v_star

    def create_payoff_matrix(self, N):

        # Узлы сетки
        x_points = np.linspace(0, 1, N+1)
        y_points = np.linspace(0, 1, N+1)

        # Инициализация матрицы нулями
        matrix = np.zeros((N+1, N+1))
        # Заполнение матрицы значениями ядра
        for i, x in enumerate(x_points):
            for j, y in enumerate(y_points):
                matrix[i, j] = self.H(x, y)

        return matrix, x_points, y_points

    def solve_by_brown_robinson(self, matrix, max_iter=200, eps=1e-6):

        m, n = matrix.shape  # размеры матрицы

        # Счётчики выбора чистых стратегий (начинаем с первой стратегии для каждого)
        x_counts = np.zeros(m)   # для игрока 1
        y_counts = np.zeros(n)   # для игрока 2
        x_counts[0] = 1
        y_counts[0] = 1

        # Переменные для отслеживания сходимости
        prev_x_star = None          # предыдущее распределение вероятностей игрока 1
        prev_y_star = None          # предыдущее распределение вероятностей игрока 2
        prev_x_idx = None           # индекс главной стратегии игрока 1 на предыдущей итерации
        prev_y_idx = None           # индекс главной стратегии игрока 2 на предыдущей итерации
        consecutive_small_errors = 0  # счётчик подряд идущих малых изменений

        for k in range(1, max_iter + 1):
            # Текущие эмпирические смешанные стратегии
            x_emp = x_counts / k
            y_emp = y_counts / k

            # Игрок 1: выбирает строку, максимизирующую ожидаемый выигрыш против y_emp
            expected_gains = np.dot(matrix, y_emp)   # вектор размера m
            best_i = np.argmax(expected_gains)

            # Игрок 2: выбирает столбец, минимизирующий ожидаемый выигрыш игрока 1
            expected_losses = np.dot(x_emp, matrix)  # вектор размера n
            best_j = np.argmin(expected_losses)

            # Обновляем счётчики (игроки "разыгрывают" выбранные чистые стратегии)
            x_counts[best_i] += 1
            y_counts[best_j] += 1

            # Обновлённые смешанные стратегии (после текущей итерации)
            x_star = x_counts / (k + 1)
            y_star = y_counts / (k + 1)

            # Индексы чистых стратегий, выбиравшихся чаще всего
            x_idx = np.argmax(x_star)
            y_idx = np.argmax(y_star)

            # Проверка критерия остановки
            if prev_x_star is not None and prev_y_star is not None:
                # Ошибка – сумма изменений вероятностей для главных стратегий
                error = (abs(x_star[x_idx] - prev_x_star[prev_x_idx]) +
                         abs(y_star[y_idx] - prev_y_star[prev_y_idx]))

                if error < eps:
                    consecutive_small_errors += 1
                    if consecutive_small_errors >= 5:
                        print(f"  Остановка на итерации {k}: ошибка < {eps} в течение 5 итераций подряд")
                        return x_idx, y_idx, k
                else:
                    consecutive_small_errors = 0

            prev_x_star = x_star.copy()
            prev_y_star = y_star.copy()
            prev_x_idx = x_idx
            prev_y_idx = y_idx

        print(f"  Достигнут максимум итераций ({max_iter})")
        return x_idx, y_idx, max_iter

    def numerical_solution(self, max_N=12):
        results = []

        for N in range(2, max_N + 1):
            # Построение платёжной матрицы и узлов сетки
            matrix, x_points, y_points = self.create_payoff_matrix(N)

            # Вывод матрицы
            print(f"\nN={N}")
            print("[")
            for i in range(N+1):
                row_str = "["
                for j in range(N+1):
                    row_str += f"{matrix[i,j]:7.3f}"
                    if j < N:
                        row_str += " "
                row_str += "]"
                print(row_str)
            print("]")

            # Поиск седловой точки в дискретной игре
            row_mins = np.min(matrix, axis=1)
            lower_price = np.max(row_mins)
            col_maxs = np.max(matrix, axis=0)
            upper_price = np.min(col_maxs)

            saddle_found = False
            saddle_i, saddle_j = -1, -1

            # Перебор всех элементов матрицы в поисках седловой точки
            for i in range(N+1):
                for j in range(N+1):
                    if (abs(matrix[i,j] - row_mins[i]) < 1e-6 and
                        abs(matrix[i,j] - col_maxs[j]) < 1e-6):
                        if abs(lower_price - upper_price) < 1e-6:
                            saddle_found = True
                            saddle_i, saddle_j = i, j
                            break
                if saddle_found:
                    break

            if saddle_found:
                x_num = x_points[saddle_i]
                y_num = y_points[saddle_j]
                v_num = matrix[saddle_i, saddle_j]
                print(f"\nЕсть седловая точка:")
                print(f"x={x_num:.3f} y={y_num:.3f} H={v_num:.3f}")
                results.append({
                    'N': N, 'method': 'седловая точка',
                    'x': x_num, 'y': y_num, 'v': v_num, 'iterations': None
                })
            else:
                # метод Брауна–Робинсона
                x_idx, y_idx, iterations = self.solve_by_brown_robinson(matrix)
                x_num = x_points[x_idx]
                y_num = y_points[y_idx]
                v_num = matrix[x_idx, y_idx]
                print(f"\nСедловой точки нет, решение методом Брауна-Робинсона:")
                print(f"x={x_num:.3f} y={y_num:.3f} H={v_num:.3f}")
                print(f"  Потребовалось итераций: {iterations}")
                results.append({
                    'N': N, 'method': 'Брауна-Робинсона',
                    'x': x_num, 'y': y_num, 'v': v_num, 'iterations': iterations
                })

        return results


def main():
    # Коэффициенты ядра (пример, можно заменить на нужный вариант)
    a, b, c, d, e = -4, 4, 8, -12/5, -28/5

    # Создание объекта-решателя
    solver = ContinuousGameSolver(a, b, c, d, e)

    # Аналитическое решение
    x_ana, y_ana, v_ana = solver.analytical_solution()

    # Численное решение для сеток от 2 до 5
    numerical_results = solver.numerical_solution(max_N=5)

    # Вывод сравнения
    print(f"\n{'='*60}")
    print(f"Аналитическое решение:    x* = {x_ana:.4f}, y* = {y_ana:.4f}, v = {v_ana:.4f}")

    # Берём последний результат (для наибольшего N)
    best = numerical_results[-1]
    print(f"Численное решение (N={best['N']}): x* = {best['x']:.4f}, y* = {best['y']:.4f}, v = {best['v']:.4f}")
    if best['iterations']:
        print(f"  Итераций потребовалось: {best['iterations']}")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()


Ядро игры: H(x,y) = -4·x² + 2·y² + 8·x·y + -0.8·x + -6.4·y

Оптимальная стратегия игрока A: x* = 0.500000
Оптимальная стратегия игрока B: y* = 0.600000
Цена игры: v = -2.120000

N=2
[
[  0.000  -2.700  -4.400]
[ -1.400  -2.100  -1.800]
[ -4.800  -3.500  -1.200]
]

Есть седловая точка:
x=0.500 y=0.500 H=-2.100

N=3
[
[  0.000  -1.911  -3.378  -4.400]
[ -0.711  -1.733  -2.311  -2.444]
[ -2.311  -2.444  -2.133  -1.378]
[ -4.800  -4.044  -2.844  -1.200]
]
  Достигнут максимум итераций (200)

Седловой точки нет, решение методом Брауна-Робинсона:
x=0.667 y=0.667 H=-2.133
  Потребовалось итераций: 200

N=4
[
[  0.000  -1.475  -2.700  -3.675  -4.400]
[ -0.450  -1.425  -2.150  -2.625  -2.850]
[ -1.400  -1.875  -2.100  -2.075  -1.800]
[ -2.850  -2.825  -2.550  -2.025  -1.250]
[ -4.800  -4.275  -3.500  -2.475  -1.200]
]

Есть седловая точка:
x=0.500 y=0.500 H=-2.100

N=5
[
[  0.000  -1.200  -2.240  -3.120  -3.840  -4.400]
[ -0.320  -1.200  -1.920  -2.480  -2.880  -3.120]
[ -0.960  -1.520  -1.920